# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. " 5 (2, 3)", создается список методом .arange затем reshape(2,3) меняет его размерность разбивая на две отдельные строки каждая по 3 элемента, далее выводим на экран элемент у которого индекст строки 1 и колонка 2 выходит элемент 5, метод shape проверяет массив на размерность которая была задана .reshape как (2, 3)

2. " 1 8 ", в созданном масиве метод argmax ищет индекс самого большого элемента, метод a.max находит сам элемент
3. "99", Переменная b создает не новый массив, а представление. Она ссылается на тот же блок памяти, что и a, с инструкцией брать только каждый третий элемент. Когда мы меняем нулевой элемент b[0] = 99, изменение происходит прямо в исходном блоке памяти. Так как a[0] указывает на эту же ячейку, значение в a тоже становится 99
4. "2, 0.5", метод sum фактически находит количество элементов больше 3 с помощью булевой логики все элементы которые не подходят условию становятся 0 а те которые подходят 1, он считает нолики и однерки и получается 2 так как только два элемента в массиве больше 3, метод .mean также считает ноли и однерки только делит еще на общее количестов элементов в массиве и находит среднее арифмитическое правильных элементов в массиве так получается 2 / 4 = 0.5  
5. "(2,3) 0.25" np.zeros(2, 3)создает двухмерный массив 2 строки на 3 столлбца а shape просто выводит размерость массива, linspace(0, 1 , 5) создает массив с 5 элементами от 0 до 1 где все точки равноудаленны то есть [0. 0.25 0.50 0.75 1.] отсюда берем индекс 1 получается 0.25 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
# База
#1
print(f"Позиция: {amount.argmax()} | Сумма: {amount.max()} | Клиент:{cust[amount.argmax()]} | Город:{city[amount.argmax()]}")

Позиция: 1001 | Сумма: 9999990.0 | Клиент:TEST | Город:Алматы


In [4]:
#2
rev = amount[paid].sum()
unpaid = amount[~paid].sum()
print(f"Сумма оплаченных: {rev} | Сумма неоплаченных: {unpaid
      }")

Сумма оплаченных: 101909360.0 | Сумма неоплаченных: 7974000.0


In [5]:
#3
shym = city == "Шымкент"
print(f" Количество заказов в Шымкенте: {shym.sum()} | Доля от всех заказов: {shym.mean() * 100}%")

 Количество заказов в Шымкенте: 804 | Доля от всех заказов: 14.71718835804503%


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [73]:
# Стандарт
#4
exp_orders = amount[paid][amount[paid] > 50000]
test = amount[paid][cust[paid] == "TEST"]
lost_rev = exp_orders.sum() * 0.15
test_1 = test.sum() * 0.15


print(f"Недополученная выручка: {lost_rev} ")
print(f"Выручка без TEST: {lost_rev - test_1}, ")

print(f"""Выручка без клиента TEST меньше в 
несколько раз потому что все три заказа являются тестовыми и 
ненастоящими поэтому сильно завышены до 9999990: {test}""")

Недополученная выручка: 5702422.5 
Выручка без TEST: 1202427.0, 
Выручка без клиента TEST меньше в 
несколько раз потому что все три заказа являются тестовыми и 
ненастоящими поэтому сильно завышены до 9999990: [9999990. 9999990. 9999990.]


In [64]:
#5
share = {}
for c in np.unique(city):
    share[c] = (days[city == c] <= 2).mean()

print(f" Доля быстрых заказов: {share}")        

 Доля быстрых заказов: {np.str_('Актобе'): np.float64(0.6221662468513854), np.str_('Алматы'): np.float64(0.6453382084095064), np.str_('Астана'): np.float64(0.6352517985611511), np.str_('Атырау'): np.float64(0.6438356164383562), np.str_('Караганда'): np.float64(0.6784386617100372), np.str_('Шымкент'): np.float64(0.29850746268656714)}


In [86]:
#6
tot = (amount[paid & (cust != "TEST")])
avg = tot.mean()
print(f"Средний чек: {round(avg)}")
print("""Ошибка была в том что код считал неоплаченные заказы и 
тестовые("TEST"), вместе со всем остальным из за чего средний чек выходил некорректным, 
в исправленном коде используются только оплаченные и настоящие заказы для расчета среднего чека""")

Средний чек: 14672
Ошибка была в том что код считал неоплаченные заказы и 
тестовые("TEST"), вместе со всем остальным из за чего средний чек выходил некорректным, 
в исправленном коде используются только оплаченные и настоящие заказы для расчета среднего чека


## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж